In [1]:
from tools.Handler import *

In [2]:
import pandas as pd
import json

In [3]:
# ---- 5. (later) load it back ----
df_loaded = pd.read_csv(r"question_answers\medium_questions_with_embeddings.csv")
df_loaded["embedding"] = df_loaded["embedding"].apply(
    lambda v: json.loads(v) if isinstance(v, str) and v else None
)

In [4]:
print("Reloaded OK. First embedding length:", len(df_loaded["embedding"].iloc[0]))

Reloaded OK. First embedding length: 768


In [5]:
df_loaded = df_loaded.rename(columns={'ideal_answer': 'answer'})

In [6]:
df_loaded['experience']="common"

In [14]:
df_loaded['difficulty']="medium"

In [15]:
df_loaded.head()

,role,question,keywords,answer,categories,weights,embedding,experience,difficulty
0,Product Manager,Tell me about a product you love. How would yo...,"user empathy, prioritization, metrics, trade-offs","I use Spotify daily, and the job it does for m...",['experience'],"[{'label': 'experience', 'score': 1.0}]","[-0.008109015, 0.039358996, -0.14959049, -0.00...",common,medium
1,Product Manager,"How would you improve [common app, e.g. YouTub...","goal clarification, user segment, prioritized ...",First I'd clarify the goal — are we optimizing...,"['technical', 'introduction']","[{'label': 'technical', 'score': 0.6}, {'label...","[-0.02850073, -0.021145083, -0.15418789, -0.00...",common,medium
2,Product Manager,Design a product for [X use case].,"clarify user, pain points, prioritize, MVP, su...",I'd start by clarifying who the user is and wh...,['technical'],"[{'label': 'technical', 'score': 1.0}]","[-0.010968673, 0.07045379, -0.11780276, -0.021...",common,medium
3,Product Manager,How would you prioritize a backlog of 10 featu...,"RICE framework, impact, effort, trade-offs, st...",I'd score each feature using a framework like ...,['situational'],"[{'label': 'situational', 'score': 1.0}]","[-0.0078732725, 0.040321097, -0.14830743, -0.0...",common,medium
4,Product Manager,Walk me through how you'd launch a new feature.,"goal setting, cross-functional alignment, phas...",I'd start with a clear goal and success metric...,['introduction'],"[{'label': 'introduction', 'score': 1.0}]","[-0.06777766, -0.0037636987, -0.1417737, -0.01...",common,medium


In [10]:
source_name="mediumsimple_questions_with_embeddings.csv"
source_description ="this dataset contains only medium questions"

#### old one

In [ ]:
import ast

def to_json_text(value):
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return "null"
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            value = ast.literal_eval(value)

    return json.dumps(value, ensure_ascii=False)

df_to_insert = df_loaded.copy()

for column in ["keywords", "categories", "weights", "embedding"]:
    if column in df_to_insert.columns:
        df_to_insert[column] = df_to_insert[column].apply(to_json_text)

batch_insert_questions(
    df_to_insert,
    source_name=source_name,
    source_description=source_description
)

#### old one

#### new one

In [ ]:
import ast
import json

def to_json_text(value):
    if value is None:
        return "null"

    if isinstance(value, str):
        value = value.strip()
        if not value:
            return "null"

        try:
            parsed = json.loads(value)
        except (TypeError, json.JSONDecodeError):
            try:
                parsed = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                parsed = value
        value = parsed

    elif hasattr(value, "tolist"):
        value = value.tolist()

    return json.dumps(value, ensure_ascii=False, default=str)

df_to_insert = df_loaded.copy()

for column in ["keywords", "categories", "weights", "embedding"]:
    if column in df_to_insert.columns:
        df_to_insert[column] = df_to_insert[column].apply(to_json_text)

batch_insert_questions(
    df_to_insert,
    source_name=source_name,
    source_description=source_description
)

📌 Using source_id: 3
📊 Total rows: 150
✅ Batch 1: 150 inserted (0–150)

🎉 Done. inserted=150, skipped=0, source_id=3


In [ ]:
# import pandas as pd

# # 1. Load CSV
# # df = pd.read_csv("your_file.csv")

# # 2. Check total rows and unique rows
# total_rows = len(df)
# unique_rows = df.drop_duplicates().shape[0]

# print(f"📊 Total rows: {total_rows}")
# print(f"🔹 Unique rows: {unique_rows}")
# print(f"🔸 Duplicate rows: {total_rows - unique_rows}")

# # 3. Show duplicate rows (if any)
# duplicates = df[df.duplicated(keep=False)]
# if not duplicates.empty:
#     print(f"\n⚠️ Found {len(duplicates)} duplicate rows (including original):")
#     print(duplicates.head(10))  # Show first 10 duplicates
# else:
#     print("\n✅ No duplicates found!")

In [2]:
from tools.context_handler import relevant_context

In [3]:
results = relevant_context("What is the capital of France?", top_k=3)

In [4]:
results

["Question: Imagine you've been tasked to migrate our custom warehouse management system from on-premises to AWS. The team lead insists on using Terraform for IaC, but you have concerns about its compatibility with our legacy codebase. Describe your approach to addressing this issue while maintaining open communication with the team and ensuring minimal disruption to our business operations.\nIdeal_Answer: This comes down to making the trade-offs explicit and then acting decisively. I'd start by getting clarity on priorities directly from stakeholders rather than assuming, then communicate trade-offs transparently to everyone affected, paying close attention to AWS, Terraform, IaC, legacy codebase since those are the constraints that actually drive the right answer here. Concretely, I'd lay out the options, flag the risks tied to AWS, and agree with stakeholders on which trade-off we're willing to accept before committing to a plan. This is the kind of situation where sticking to the f